# QESEM: A Qiskit Function by Qedma

Run quantum circuits on noisy QPUs to obtain highly accurate error-free results with efficient QPU-time overheads.

**Features:** Guaranteed accuracy, scalable to 133+ qubits, application-agnostic, extended gate set (fractional Rzz), multibase observables.

**References:**
- [API reference](https://quantum.cloud.ibm.com/docs/api/functions/qedma-qesem)
- [Reliable high-accuracy error mitigation (arXiv:2508.10997)](https://arxiv.org/abs/2508.10997)
- [Support: support@qedma.com](mailto:support@qedma.com)

## 1. Connection & Setup

In [1]:
import os
import qiskit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_catalog import QiskitFunctionsCatalog
from qiskit_ibm_runtime import QiskitRuntimeService

ibm_token = os.environ.get("IBM_KEY")
ibm_instance = os.environ.get("IBM_INSTANCE_CRN")

ibm_token = "rEq7896NH0FHWoU1uw1CASkzG_crj6hhr-veMjhREtlW"
ibm_instance = "crn:v1:bluemix:public:quantum-computing:us-east:a/a7e60db529fb4fcd9049e3ab9d7176cf:903fac89-2d9a-44c0-b96a-ad9dddf8dbe9::"
backend_name = "ibm_miami"

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=ibm_token, instance=ibm_instance)
print("Backends:", service.backends())

catalog = QiskitFunctionsCatalog(instance=ibm_instance, token=ibm_token)
print("Functions:", catalog.list())

qesem_function = catalog.load("qedma/qesem")
print("QESEM loaded successfully")

qiskit_runtime_service._discover_account:WARNING:2026-04-17 14:05:13,604: Loading account with the given token. A saved account will not be used.


Backends: [<IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_kingston')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>]
Functions: [QiskitFunction(global-data-quantum/quantum-portfolio-optimizer), QiskitFunction(qedma/qesem), QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]
QESEM loaded successfully


## 2. Circuit & Observables

In [2]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

print(circ)
print("Observables:", avg_magnetization.num_qubits, "qubits")

               
q_0: ──■───────
     ┌─┴─┐     
q_1: ┤ X ├──■──
     └───┘┌─┴─┐
q_2: ──■──┤ X ├
     ┌─┴─┐└───┘
q_3: ┤ X ├──■──
     └───┘┌─┴─┐
q_4: ─────┤ X ├
          └───┘
Observables: 5 qubits


## 3. QPU Time Estimation (free, no execution)

In [ ]:
time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={"estimate_time_only": "analytical"},
    backend_name=backend_name,
)
print(f"initialicing QPU time")
time_result = time_estimation_job.result()
print(f"algo mas")
print(f"Estimated QPU time: {time_result[0].metadata}")

## 4. Execute with QESEM

⚠️ **WARNING**: This consumes QPU time. Uncomment the cell below only when ready.

In [ ]:
exec_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,
)
# print("Status:", exec_job.id())
# then email the tea to ask why is failing about this issue.
print("Status:", exec_job.status())
result = exec_job.result()

Status: QUEUED


## 5. Retrieve Results

Uncomment after execution completes.

In [ ]:
pub_result = result[0]
print(f"Mitigated expectation values: {pub_result.data.evs}")
print(f"Mitigated error-bar: {pub_result.data.stds}")

noisy = pub_result.metadata["noisy_results"]
print(f"Noisy expectation values: {noisy.evs}")
print(f"Noisy error-bar: {noisy.stds}")

print(f"Total QPU time: {pub_result.metadata['total_qpu_time']}")
print(f"Gate fidelities: {pub_result.metadata['gate_fidelities']}")
print(f"Total/mitigation shots: {pub_result.metadata['total_shots']} / {pub_result.metadata['mitigation_shots']}")

for i, tc in enumerate(pub_result.metadata["transpiled_circs"]):
    print(f"Circuit {i}: qubits={tc['qubit_map']}, bases={tc['num_measurement_bases']}")